# Multi-Target Clustering & Feature Extraction

UMAP + HDBSCAN cluster enrichment and LASSO bridge for the three displacement vectors or the raw context embeddings.

TARGET | Vector | Question |
- context | $z_{ctx}$ | What patient-state clusters exist?
- delta | P − C (z_{pred} - z_{ctx}) | What types of predicted change cluster together?
- pred_error | P − T (z_{pred} - z_{tgt}) | Where does the predictor systematically fail?

Output dir: `experiments/{MODEL_TAG}/clusters_{TARGET}/`.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

import numpy as np
import json

from src.utils.io import EXPERIMENTS_DIR, PROCESSED_DIR, load_metadata
from src.utils.seed import load_seed, set_global_seed
from src.analysis.geometry import fit_tform_umap_2d
from src.analysis.clustering import (
    broadcast_to_samples,
    pool_to_patients,
    run_cluster_enrichment,
    run_lasso_bridge,
)
from src.analysis.plotting import (
    show_or_savefig,
    plot_context_umap,
    plot_cluster_umap,
    plot_cluster_enrichment,
    plot_lasso_heatmap,
    plot_r2_bar,
)

In [ ]:
# -- Settings --
MODEL_TAG        = "test_01"
EMB_NAME         = "embeddings_40.npz"
TARGET           = "context"       # "context" | "delta" | "pred_error"
MIN_CLUSTER_SIZE = 10
TOP_K            = 10
SAVE_FIGS        = False
SAVE_DATA        = True

exp_dir      = EXPERIMENTS_DIR / MODEL_TAG
set_global_seed(load_seed(exp_dir))

emb_path     = exp_dir / "embeddings" / EMB_NAME
geo_dir      = exp_dir / "geometry"
clusters_dir = exp_dir / f"clusters_{TARGET}"
fig_dir      = clusters_dir / "figures"

def _sp(name: str):
    return fig_dir / name if SAVE_FIGS else None

In [ ]:
# -- Load embeddings --
npz = np.load(emb_path, allow_pickle=True)
z_context   = npz["z_context"]
z_pred      = npz["z_pred"]
z_target    = npz["z_target"]
labels      = npz["labels"]
subject_ids = npz["subject_ids"]

**Select Target Vector**

In [ ]:
if TARGET == "context":
    target_vec = z_context
elif TARGET == "delta":
    target_vec = npz["delta"] if "delta" in npz else z_pred - z_context
elif TARGET == "pred_error":
    target_vec = npz["pred_error"] if "pred_error" in npz else z_pred - z_target
else:
    raise ValueError(f"Unknown TARGET: {TARGET!r}")

N, D = target_vec.shape
print(f"TARGET = {TARGET!r}  |  shape = ({N}, {D})")
print(f"  ||vec|| mean = {np.linalg.norm(target_vec, axis=-1).mean():.4f}")

In [ ]:
# -- Load metadata and broadcast to sample level --
metadata_patients, feature_names, patient_ids = load_metadata(PROCESSED_DIR)
metadata_samples = broadcast_to_samples(metadata_patients, patient_ids, subject_ids)
print(f"Metadata: {metadata_patients.shape[0]} patients x {len(feature_names)} features "
      f"-> {metadata_samples.shape[0]} samples")

### UMAP

In [ ]:
# For TARGET="context", reuse geometry/context_umap.npy if it exists
cached_umap = geo_dir / "context_umap.npy"
if TARGET == "context" and cached_umap.exists():
    umap_emb = np.load(cached_umap)
    print(f"Loaded cached UMAP from {cached_umap}")
else:
    metric = "euclidean" if TARGET == "context" else "cosine"
    print(f"Computing UMAP (metric={metric!r}) ...")
    umap_emb = fit_tform_umap_2d(target_vec, metric=metric)
    print("Done.")

if SAVE_DATA:
    clusters_dir.mkdir(parents=True, exist_ok=True)
    np.save(clusters_dir / "umap_embedding.npy", umap_emb)
    print(f"Saved UMAP to {clusters_dir / 'umap_embedding.npy'}")

### HBDSCAN + Enrichment

In [ ]:
cluster_result = run_cluster_enrichment(
    umap_emb, metadata_samples, feature_names,
    min_cluster_size=MIN_CLUSTER_SIZE,
)

In [ ]:
print(f"Clusters: {cluster_result['n_clusters']}  |  Noise: {cluster_result['n_noise']}")
for cid in cluster_result["cluster_ids"]:
    size = cluster_result["cluster_sizes"].get(cid, 0)
    profile = cluster_result["cluster_profiles"].get(cid, [])
    top_feat = profile[0]["feature"] if profile else "—"
    print(f"  C{cid}: n={size:<5}  top feature = {top_feat}")

if cluster_result.get("unlabeled_clusters"):
    print(f"Unlabeled clusters (no |z| > 1): {cluster_result['unlabeled_clusters']}")
    
if SAVE_DATA:
    clusters_dir.mkdir(parents=True, exist_ok=True)
    np.save(clusters_dir / "cluster_labels.npy", cluster_result["cluster_labels"])
    np.save(clusters_dir / "enrichment_matrix.npy", cluster_result["enrichment_matrix"])

    summary = {
        k: v for k, v in cluster_result.items()
        if k not in ("cluster_labels", "enrichment_matrix")
    }
    # Convert numpy types for JSON serialisation
    summary["cluster_ids"] = [int(c) for c in summary.get("cluster_ids", [])]
    summary["cluster_sizes"] = {str(k): int(v) for k, v in summary.get("cluster_sizes", {}).items()}
    summary["unlabeled_clusters"] = [int(c) for c in summary.get("unlabeled_clusters", [])]
    with open(clusters_dir / "cluster_summary.json", "w") as f:
        json.dump(summary, f, indent=2, default=str)

**Cluster Plotting**

In [ ]:
cluster_labels = cluster_result["cluster_labels"]

if TARGET == "context":
    plot_context_umap(umap_emb, labels, cluster_labels,
                      show=True, save_path=_sp("context_umap_clusters.png"))
else:
    plot_cluster_umap(umap_emb, cluster_labels,
                      show=True, save_path=_sp(f"{TARGET}_umap_clusters.png"))

In [ ]:
plot_cluster_enrichment(cluster_result,
                        show=True, save_path=_sp("cluster_enrichment_heatmap.png"))

### LASSO Bridge (Tier 1)

Regress clinical metadata against PCA projections of the target vector. 

*Skipped for `"context"` since geometry.py doesn't compute PCA on z_context (it's not a displacement vector).*

In [ ]:
proj_path = geo_dir / f"{TARGET}_projections.npy"

if TARGET == "context":
    print("LASSO bridge skipped for TARGET='context'")
elif not proj_path.exists():
    print(f"LASSO bridge skipped: {proj_path} not found. (run notebooks/geometry.py first)")
else:
    pc_proj_samples = np.load(proj_path)
    pc_proj_patients = pool_to_patients(pc_proj_samples, subject_ids, patient_ids)
    print(f"PC projections: {pc_proj_samples.shape} samples -> "
          f"{pc_proj_patients.shape} patients")

    lasso_result = run_lasso_bridge(
        pc_proj_patients, metadata_patients, feature_names,
        top_k=TOP_K,
    )

    print(f"\nMean R² = {lasso_result['mean_r2']:.3f}  |  "
          f"Unexplained = {lasso_result['unexplained_variance_fraction']:.1%}")
    for pc, r2 in lasso_result["r2_per_pc"].items():
        print(f"  {pc}: R² = {r2:.3f}")

    plot_lasso_heatmap(lasso_result, save_path=_sp(f"{TARGET}_lasso_heatmap.png"))
    plot_r2_bar(lasso_result, save_path=_sp(f"{TARGET}_r2_bar.png"))

    if SAVE_DATA:
        lasso_out = {k: v for k, v in lasso_result.items()
                     if not isinstance(v, np.ndarray)}
        with open(clusters_dir / f"{TARGET}_lasso_bridge.json", "w") as f:
            json.dump(lasso_out, f, indent=2, default=str)

---

### Label Clustering Summary

In [ ]:
print(f"\n(LASSO):")
print(f"  Mean R²:     {lasso_result['mean_r2']:.3f}")
print(f"  Unexplained: {lasso_result['unexplained_variance_fraction']:.1%}")
print(f"{'='*60}")
print(f"\n(Clusters):")
print(f"  Clusters:    {cluster_result['n_clusters']}")
print(f"  Noise:       {cluster_result['n_noise']} samples")
print(f"  Unlabeled:   {len(cluster_result['unlabeled_clusters'])} clusters")